<a href="https://colab.research.google.com/github/JoshuaNiel/CS452/blob/main/mongo/Step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3 — Aggregation Pipelines: `$unwind`, `$lookup`, `$project`, and `$group`

In this activity we'll work through two guided examples that combine multiple aggregation stages.
By the end you should be comfortable with:

| Stage | What it does |
|---|---|
| `$unwind` | Flatten an array field into one doc per element |
| `$lookup` | Left-outer-join another collection |
| `$project` | Reshape documents — rename, compute, include/exclude fields |
| `$group` | Aggregate values (sum, avg, count, first, etc.) |

These are exactly the tools you'll need for the Step 5 COVID homework.

> **FYI:** There's also a `$out` stage that writes pipeline results to a **new collection** — like `CREATE TABLE AS SELECT` in SQL. We won't use it here since we're on a shared read-only connection, but you'll want it when working on your own Atlas cluster.

---
## Setup

In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.9 MB/s eta 0:00:00


In [2]:
import pymongo

user = "class"
password = "184vLpDKvOhvv528"
cluster = "cluster0"
dnsprefix = "wvdjn"
connectionUrl = f"mongodb+srv://{user}:{password}@{cluster}.{dnsprefix}.mongodb.net/"
client = pymongo.MongoClient(connectionUrl)
print(f"Ping result: {client.admin.command('ping')}")

db = client.movies
client.admin.command('ping')
print("Connected!")

Ping result: {'ok': 1}
Connected!


---
## Example 1 — `$unwind` + `$group`: Count actors per genre

**Goal:** For each genre, count how many actor appearances there are and find the average cast size.

Each movie document has an embedded `actors` array. `$unwind` explodes that array
so we get **one document per actor appearance**, then we group by genre.

In [3]:
pipeline = [
    # Step 1: Unwind the embedded actors array — one doc per actor per movie
    {"$unwind": "$actors"},
    # Step 2: Group by genre, count actor appearances and distinct movies
    {"$group": {
        "_id": "$genre",
        "total_actor_appearances": {"$sum": 1},
        "movies_with_actors":      {"$addToSet": "$title"}
    }},
    # Step 3: Project to compute average cast size
    {"$project": {
        "_id": 0,
        "genre": "$_id",
        "total_actor_appearances": 1,
        "movie_count": {"$size": "$movies_with_actors"},
        "avg_cast_size": {
            "$round": [{"$divide": ["$total_actor_appearances", {"$size": "$movies_with_actors"}]}, 1]
        }
    }},
    {"$sort": {"total_actor_appearances": -1}}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

{'total_actor_appearances': 58, 'genre': 'drama', 'movie_count': 20, 'avg_cast_size': 2.9}
{'total_actor_appearances': 41, 'genre': 'Science-fiction', 'movie_count': 13, 'avg_cast_size': 3.2}
{'total_actor_appearances': 40, 'genre': 'Action', 'movie_count': 13, 'avg_cast_size': 3.1}
{'total_actor_appearances': 35, 'genre': 'crime', 'movie_count': 11, 'avg_cast_size': 3.2}
{'total_actor_appearances': 22, 'genre': 'Thriller', 'movie_count': 7, 'avg_cast_size': 3.1}
{'total_actor_appearances': 11, 'genre': 'Comédie', 'movie_count': 4, 'avg_cast_size': 2.8}
{'total_actor_appearances': 10, 'genre': 'Western', 'movie_count': 4, 'avg_cast_size': 2.5}
{'total_actor_appearances': 9, 'genre': 'Guerre', 'movie_count': 2, 'avg_cast_size': 4.5}
{'total_actor_appearances': 7, 'genre': 'Horreur', 'movie_count': 4, 'avg_cast_size': 1.8}
{'total_actor_appearances': 5, 'genre': 'Fantastique', 'movie_count': 2, 'avg_cast_size': 2.5}
{'total_actor_appearances': 5, 'genre': 'romance', 'movie_count': 1, 'av

### Key takeaways
- `$unwind` turns an **embedded array** into separate documents — like normalizing on the fly.
- `$addToSet` collects unique values during a `$group` (we used it to count distinct movies).
- `$project` can compute new fields with expressions like `$divide`, `$size`, and `$round`.
- Stages execute **in order** — a pipeline is just a list of transformations.

---
## Example 2 — `$lookup` + `$unwind` + `$project`: Flatten movies with ratings

**Goal:** Produce a clean, flat view of each movie with its `title`, `genre`, `country`, and `rating`.

The `ratings` collection is a separate collection that stores `movie_id` and `rating`.
We need `$lookup` to join it onto `movies` — this is the MongoDB equivalent of a SQL `SELECT ... JOIN ... ON ...`.

In [4]:
pipeline = [
    # Join ratings onto movies (ratings.movie_id matches movies._id)
    {"$lookup": {
        "from": "ratings",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "rating_info"
    }},
    # preserveNullAndEmptyArrays = True keeps movies even if no rating (LEFT JOIN)
    {"$unwind": {"path": "$rating_info", "preserveNullAndEmptyArrays": True}},
    # Clean output — only the fields we want, with a fallback for missing ratings
    {"$project": {
        "_id": 0,
        "title": 1,
        "genre": 1,
        "country": 1,
        "rating": {"$ifNull": ["$rating_info.rating", "N/A"]}
    }}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

{'title': 'Piège de cristal', 'genre': 'Action', 'country': 'USA', 'rating': 2}
{'title': '58 minutes pour vivre', 'genre': 'Action', 'country': 'USA', 'rating': 1}
{'title': 'Bad Lieutenant', 'genre': 'drama', 'country': 'USA', 'rating': 3.5}
{'title': 'Le parrain', 'genre': 'drama', 'country': 'USA', 'rating': 3}
{'title': 'Le parrain III', 'genre': 'drama', 'country': 'USA', 'rating': 20}
{'title': 'Sixième sens', 'genre': 'Fantastique', 'country': 'USA', 'rating': 1}
{'title': 'The Dark Knight', 'genre': 'Science-fiction', 'country': 'USA', 'rating': 5}
{'title': 'Les oiseaux', 'genre': 'Horreur', 'country': 'USA', 'rating': 4}
{'title': 'Django unchained', 'genre': 'Western', 'country': 'USA', 'rating': 4.5}
{'title': 'American Beauty', 'genre': 'Comédie', 'country': 'USA', 'rating': 2}
{'title': 'Impitoyable', 'genre': 'Western', 'country': 'USA', 'rating': 1}
{'title': 'Une journée en enfer', 'genre': 'Action', 'country': 'USA', 'rating': 1}
{'title': 'Mary à tout prix', 'genre'

### Key takeaways
- `$lookup` joins a **separate collection** by matching field values — like a SQL JOIN.
- `preserveNullAndEmptyArrays: True` makes `$unwind` behave like a **LEFT JOIN** instead of an INNER JOIN.
- `$ifNull` provides a default when a field is missing — handy for keeping output clean.
- This "join → unwind → project" pattern is one you'll use constantly.

---
## Now you try!

Use the `movies` and `ratings` collections for the exercises below.

### Exercise 1

Find the **top-rated movie per genre**.

Hints:
- `$lookup` ratings onto movies
- `$unwind` the ratings
- `$project` to keep `title`, `genre`, and `rating`
- `$sort` by rating descending, then `$group` by genre with `$first`

In [15]:
# your code here
pipeline = [
    {
        "$lookup": {
            "from": "ratings",
            "localField": "_id",
            "foreignField": "movie_id",
            "as": "rating_info"
        }
    },
    {
        "$unwind": {
            "path": "$rating_info",
        }
    },
    {
        "$sort": {
            "rating_info.rating": -1
        }
    },
    {
        "$group": {
            "_id": "$genre",
            "title": {"$first": "$title"},
            "top_rating": {
                "$first": "$rating_info.rating"
            },
            "top_movie": {
                "$first": "$title"
            }
        }
    },
        {
        "$project": {
            "_id": 0,
            "genre": "$_id",
            "title": 1,
            "top_rating": 1,
            "top_movie": 1,
        }
    },
]

res1 = db.movies.aggregate(pipeline)
list(res1)

[{'title': 'Batman begins',
  'top_rating': 5,
  'top_movie': 'Batman begins',
  'genre': 'Thriller'},
 {'title': 'Django unchained',
  'top_rating': 4.5,
  'top_movie': 'Django unchained',
  'genre': 'Western'},
 {'title': 'Lost in Translation',
  'top_rating': 4.5,
  'top_movie': 'Lost in Translation',
  'genre': 'romance'},
 {'title': 'Sleepy Hollow',
  'top_rating': 3.5,
  'top_movie': 'Sleepy Hollow',
  'genre': 'Fantastique'},
 {'title': 'Le gendarme et les extra-terrestres',
  'top_rating': 4.5,
  'top_movie': 'Le gendarme et les extra-terrestres',
  'genre': 'Comédie'},
 {'title': 'Le parrain III',
  'top_rating': 20,
  'top_movie': 'Le parrain III',
  'genre': 'drama'},
 {'title': 'Inglourious Basterds',
  'top_rating': 5,
  'top_movie': 'Inglourious Basterds',
  'genre': 'Guerre'},
 {'title': 'Blade Runner',
  'top_rating': 5,
  'top_movie': 'Blade Runner',
  'genre': 'Action'},
 {'title': 'La mort aux trousses',
  'top_rating': 1.5,
  'top_movie': 'La mort aux trousses',
  '

### Exercise 2

For each genre, compute:
- `movie_count` — number of movies
- `avg_rating` — average rating
- `avg_cast_size` — average number of actors

Sort by `avg_rating` descending.

Hints:
- `$lookup` ratings onto movies
- `$unwind` the ratings array
- `$project` to pull out `rating` and compute `cast_size` using `$size` on the `actors` array
- `$group` by genre with `$avg` and `$sum`
- Use `$project` at the end to rename `_id` to `genre` and round the averages

In [20]:
# your code here
pipeline = [
        {
        "$lookup": {
            "from": "ratings",
            "localField": "_id",
            "foreignField": "movie_id",
            "as": "rating_info"
        }
    },
    {
        "$unwind": {
            "path": "$rating_info",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$project": {
            "_id": 0,
            "title": 1,
            "rating": "$rating_info.rating",
            "cast_size": {"$size": "$actors"},
            "genre": 1,

        }
    },
    {
        "$group": {
            "_id": "$genre",
            "movie_count": {"$sum": 1},
            "avg_rating": {"$avg": "$rating"},
            "avg_cast_size": {"$avg": "$cast_size"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "genre": "$_id",
            "movie_count": 1,
            "avg_rating": {"$round": ["$avg_rating", 1]},
            "avg_cast_size": {"$round": ["$avg_cast_size", 1]}
        }
    }

]
res2 = db.movies.aggregate(pipeline)
list(res2)

[{'movie_count': 22,
  'genre': 'drama',
  'avg_rating': 3.8,
  'avg_cast_size': 2.6},
 {'movie_count': 13,
  'genre': 'Action',
  'avg_rating': 3.2,
  'avg_cast_size': 3.1},
 {'movie_count': 4,
  'genre': 'Comédie',
  'avg_rating': 2.8,
  'avg_cast_size': 2.8},
 {'movie_count': 2,
  'genre': 'Fantastique',
  'avg_rating': 2.2,
  'avg_cast_size': 2.5},
 {'movie_count': 5,
  'genre': 'Western',
  'avg_rating': 3.0,
  'avg_cast_size': 2.0},
 {'movie_count': 1,
  'genre': 'romance',
  'avg_rating': 4.5,
  'avg_cast_size': 5.0},
 {'movie_count': 7,
  'genre': 'Thriller',
  'avg_rating': 3.6,
  'avg_cast_size': 3.1},
 {'movie_count': 13,
  'genre': 'Science-fiction',
  'avg_rating': 4.5,
  'avg_cast_size': 3.2},
 {'movie_count': 4,
  'genre': 'Horreur',
  'avg_rating': 4.0,
  'avg_cast_size': 1.8},
 {'movie_count': 2,
  'genre': 'Suspense',
  'avg_rating': 1.2,
  'avg_cast_size': 1.5},
 {'movie_count': 11,
  'genre': 'crime',
  'avg_rating': 3.3,
  'avg_cast_size': 3.2},
 {'movie_count': 4,